# Lecture 28 — Matrix Backpropagation: dW, db, dX by Hand

Read the story first: [`blog.md`](<blog.md>). This lab uses the same two-student, three-feature example.


## Step 1 — Problem

A dense layer computes $Z=XW+b$. How can we compute every gradient without differentiating each weight separately?

## Step 2 — Prediction

Before running code, predict the shapes of `dW`, `db`, and `dX` and the first entry of `dW`.

In [ ]:
import numpy as np

X = np.array([[1., 2., 3.], [2., 1., 4.]])
W = np.array([[1., 0.], [0., 2.], [1., 1.]])
b = np.array([1., 2.])
dZ = np.array([[1., 2.], [-1., 3.]])

Z = X @ W + b
print(Z)


## Step 4 — Mathematics

The layer is $Z=XW+b$. The gradients should be $dW=X^TdZ$, $db=\sum dZ$, and $dX=dZW^T$.


In [ ]:
dW = X.T @ dZ
db = dZ.sum(axis=0)
dX = dZ @ W.T

assert dW.shape == W.shape
assert db.shape == b.shape
assert dX.shape == X.shape

np.set_printoptions(suppress=True)
print('dW =\n', dW)
print('db =', db)
print('dX =\n', dX)

assert np.allclose(dW, np.array([[-1., 8.], [1., 7.], [-1., 18.]]))
assert np.allclose(db, np.array([0., 5.]))
assert np.allclose(dX, np.array([[1., 4., 3.], [-1., 6., 2.]]))


In [ ]:
# Step 6 — first implementation: explicit loops
dW_loop = np.zeros_like(W)
for i in range(X.shape[1]):
    for j in range(W.shape[1]):
        for r in range(X.shape[0]):
            dW_loop[i, j] += X[r, i] * dZ[r, j]

db_loop = np.zeros_like(b)
for r in range(dZ.shape[0]):
    db_loop += dZ[r]

dX_loop = np.zeros_like(X)
for r in range(X.shape[0]):
    for i in range(X.shape[1]):
        for j in range(W.shape[1]):
            dX_loop[r, i] += dZ[r, j] * W[i, j]

assert np.allclose(dW_loop, dW)
assert np.allclose(db_loop, db)
assert np.allclose(dX_loop, dX)
print('Loop and matrix formulas agree.')


In [ ]:
# Step 7 — visualization: gradient magnitudes
import matplotlib.pyplot as plt

plt.figure()
plt.bar(range(dW.size), np.abs(dW.ravel()))
plt.xlabel('weight index')
plt.ylabel('|gradient|')
plt.title('Magnitude of weight gradients')
plt.show()


In [ ]:
# Step 8 — controlled experiment
# Change exactly one variable: double the first feature for the first example.
X_changed = X.copy()
X_changed[0, 0] *= 2
dW_changed = X_changed.T @ dZ

print('original dW =\n', dW)
print('changed  dW =\n', dW_changed)


In [ ]:
# Step 9 — compare the change
delta = dW_changed - dW
print('delta =\n', delta)
# Only row 0 of dW can change because only feature 0 changed.
assert np.allclose(delta[1:], 0)


## Step 10 — Observe

Only the gradients associated with the changed input feature moved. Compare this with your prediction.

## Step 11 — Explain

Each weight gradient is an input value multiplied by a downstream gradient and summed over the batch, which is exactly what $X^TdZ$ computes.

In [ ]:
# Step 12 — challenge
# YOUR CODE HERE
# 1. Remove the bias and derive the corresponding backward equations.
# 2. Create a 3-example batch and verify all three gradients with finite differences.
# 3. Intentionally try a wrong dX formula and use shape checks to reject it.


## Step 13 — Reflection

Mastery checklist: explain why `dW` uses `X.T`; explain why `db` sums over the batch; explain why `dX` uses `W.T`; and use shapes to catch a wrong formula.

**Next:** tensors generalize these same ideas beyond two dimensions.